In [1]:
# python
import sys
import os
import importlib
# columnar analysis
from coffea import processor
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema
import awkward as ak
from dask.distributed import Client, performance_report
# local
sidm_path = str(os.getcwd()).split("/sidm")[0]
# sidm_path = str(sys.path[0]).split("/sidm")[0]
if sidm_path not in sys.path: sys.path.insert(1, sidm_path)
from sidm.tools import utilities, sidm_processor, scaleout, cutflow
from sidm.tools import llpnanoaodschema
# always reload local modules to pick up changes during development
importlib.reload(utilities)
importlib.reload(sidm_processor)
importlib.reload(scaleout)
# plotting
import matplotlib.pyplot as plt
utilities.set_plot_style()
%matplotlib inline
from tqdm.notebook import tqdm
import coffea.util
import numpy as np
import mplhep as hep
import yaml

In [2]:
client = scaleout.make_dask_client("tls://localhost:8786")
client

Connection method: Direct,
Dashboard: /user/dongyub.lee@cern.ch/proxy/8787/status,
Comm: tls://192.168.197.244:8786,Workers: 0
Dashboard: /user/dongyub.lee@cern.ch/proxy/8787/status,Total threads: 0
Started: 2 minutes ago,Total memory: 0 B


In [8]:
runner = processor.Runner(
    # executor=processor.IterativeExecutor(),
    executor=processor.DaskExecutor(client=client),
    # schema=NanoAODSchema,
    schema = llpnanoaodschema.LLPNanoAODSchema,
    # maxchunks=1,
    # chunksize=100000,
    skipbadfiles=True,
)

In [9]:
channels = [
    # "data_control_region_1muLj_1egmLj",                     # MuLj == 1 and egmLj == 1 CR
    # "data_control_region_1muLj_1egmLj_cosmic_veto",       # MuLj == 1 and egmLj == 1 + Cosmic veto CR
    # "data_control_region_2muLj",                          # MuLj == 2 CR
    "data_control_region_2muLj_cosmic_veto",              # MuLj == 2 + Cosmic veto CR
]

p = sidm_processor.SidmProcessor(
    channels,
    ["muon_base", "dsaMuon_base", "mu_lj_base", "mu_matched_jet_base"],          # Muon categories
    # ["electron_base", "photon_base", "egm_lj_base", "egm_matched_jet_base"],   # EGM categories
    # ["muon_base", "dsaMuon_base", "mu_lj_base", "mu_matched_jet_base", "electron_base", "photon_base", "egm_lj_base", "egm_matched_jet_base"],   # Both categories
    unweighted_hist=False,
)

# 2Mu2E Processing

In [8]:
yaml_file_path = '../../configs/ntuples/signal_2mu2e_v10.yaml'
# Open and read the YAML file
with open(yaml_file_path, 'r') as file:
    data = yaml.safe_load(file)
signals_2mu2e_all = list(data["llpNanoAOD_v2"]["samples"].keys())

max_files_2mu2e = -1
fileset_2mu2e = utilities.make_fileset(signals_2mu2e_all, "llpNanoAOD_v2", max_files=max_files_2mu2e, location_cfg="signal_2mu2e_v10.yaml")
output_2mu2e = runner.run(fileset_2mu2e, treename="Events", processor_instance=p)
coffea.util.save(output_2mu2e, f"signal_2mu2e.coffea")

2Mu2E_100GeV_0p25GeV_0p02mm is simulation. Scaling histograms or cutflows according to lumi*xs.
Signal not in xs cfg, assuming 1fb
2Mu2E_100GeV_1p2GeV_9p6mm is simulation. Scaling histograms or cutflows according to lumi*xs.
Signal not in xs cfg, assuming 1fb
2Mu2E_100GeV_1p2GeV_0p96mm is simulation. Scaling histograms or cutflows according to lumi*xs.
Signal not in xs cfg, assuming 1fb
2Mu2E_100GeV_1p2GeV_0p096mm is simulation. Scaling histograms or cutflows according to lumi*xs.
Signal not in xs cfg, assuming 1fb
2Mu2E_100GeV_0p25GeV_20p0mm is simulation. Scaling histograms or cutflows according to lumi*xs.
Signal not in xs cfg, assuming 1fb
2Mu2E_100GeV_0p25GeV_10p0mm is simulation. Scaling histograms or cutflows according to lumi*xs.
Signal not in xs cfg, assuming 1fb
2Mu2E_100GeV_0p25GeV_2p0mm is simulation. Scaling histograms or cutflows according to lumi*xs.
Signal not in xs cfg, assuming 1fb
2Mu2E_100GeV_0p25GeV_0p2mm is simulation. Scaling histograms or cutflows according to l

# 4Mu Processing

In [9]:
yaml_file_path = '../../configs/ntuples/signal_4mu_v10.yaml'
# Open and read the YAML file
with open(yaml_file_path, 'r') as file:
    data = yaml.safe_load(file)
signals_4mu_all = list(data["llpNanoAOD_v2"]["samples"].keys())

max_files_4mu = -1
fileset_4mu = utilities.make_fileset(signals_4mu_all, "llpNanoAOD_v2", max_files=max_files_4mu, location_cfg="signal_4mu_v10.yaml")
output_4mu = runner.run(fileset_4mu, treename="Events", processor_instance=p)
coffea.util.save(output_4mu, f"signal_4mu.coffea")

4Mu_100GeV_0p25GeV_0p02mm is simulation. Scaling histograms or cutflows according to lumi*xs.
Signal not in xs cfg, assuming 1fb
4Mu_1000GeV_1p2GeV_0p96mm is simulation. Scaling histograms or cutflows according to lumi*xs.
Signal not in xs cfg, assuming 1fb
4Mu_1000GeV_5p0GeV_40p0mm is simulation. Scaling histograms or cutflows according to lumi*xs.
Signal not in xs cfg, assuming 1fb
4Mu_1000GeV_5p0GeV_20p0mm is simulation. Scaling histograms or cutflows according to lumi*xs.
Signal not in xs cfg, assuming 1fb
4Mu_1000GeV_5p0GeV_4p0mm is simulation. Scaling histograms or cutflows according to lumi*xs.
Signal not in xs cfg, assuming 1fb
4Mu_1000GeV_5p0GeV_0p4mm is simulation. Scaling histograms or cutflows according to lumi*xs.
Signal not in xs cfg, assuming 1fb
4Mu_1000GeV_5p0GeV_0p04mm is simulation. Scaling histograms or cutflows according to lumi*xs.
Signal not in xs cfg, assuming 1fb
4Mu_1000GeV_1p2GeV_9p6mm is simulation. Scaling histograms or cutflows according to lumi*xs.
Signal

# Background Processing

In [10]:
samples_bkg = [
    # "TTJets",
    
    # "QCD_Pt15To20",
    # "QCD_Pt20To30",
    # "QCD_Pt30To50",
    # "QCD_Pt50To80",
    # "QCD_Pt80To120", 
    # "QCD_Pt120To170",
    # "QCD_Pt170To300", 
    # "QCD_Pt300To470",
    # "QCD_Pt470To600", 
    # "QCD_Pt600To800", 
    # "QCD_Pt800To1000",
    # "QCD_Pt1000", 
    
    # "DYJetsToMuMu_M10to50",
    # "DYJetsToMuMu_M50", 

    "WW",
    "WZ",
    "ZZ",

]

In [11]:
fileset = utilities.make_fileset(samples_bkg[0:60], 
                                 "skimmed_llpNanoAOD_v2", 
                                 location_cfg="backgrounds.yaml",
                                 max_files = -1,
                                )

In [12]:
out = {}
for i, bkg in enumerate(samples_bkg):
    
    print(f"Processing {bkg}")
    fileset_one_bkg = {samples_bkg[i]:fileset.get(samples_bkg[i])}
    
    output = runner.run(fileset_one_bkg, treename='Events', processor_instance=p)

    #Add this sample's output to the out variable
    out[bkg] = output["out"][bkg]

    #Save output to a file!!
    out_file_name = bkg + ".coffea"
    coffea.util.save(output,out_file_name)

Processing WW
WW is simulation. Scaling histograms or cutflows according to lumi*xs..3s
Processing WZ
WZ is simulation. Scaling histograms or cutflows according to lumi*xs..3s
Processing ZZ
ZZ is simulation. Scaling histograms or cutflows according to lumi*xs..3s


# Data Processing

In [9]:
samples_data= [ 
    "DoubleMuon_2018C",
    
    "DoubleMuon_2018A_0",
    "DoubleMuon_2018A_1",
    "DoubleMuon_2018A_2",

    ## B/D is not ready yet. I'll add them as soon as they are ready
    
    # "DoubleMuon_2018D_0",
    # "DoubleMuon_2018D_1",
    # "DoubleMuon_2018D_2",
    # "DoubleMuon_2018D_3",
    # "DoubleMuon_2018D_4",
    # "DoubleMuon_2018D_5",
    # "DoubleMuon_2018D_6",
    # "DoubleMuon_2018D_7",
    # "DoubleMuon_2018D_8",
    # "DoubleMuon_2018D_9",
    # "DoubleMuon_2018D_10",
    # "DoubleMuon_2018D_11",
    # "DoubleMuon_2018D_12",
    # "DoubleMuon_2018D_13",
    # "DoubleMuon_2018D_14",
    # "DoubleMuon_2018D_15",
    # "DoubleMuon_2018D_16",
    # "DoubleMuon_2018D_17",
    # "DoubleMuon_2018D_18",
    # "DoubleMuon_2018D_19",
    # "DoubleMuon_2018D_20",
    # "DoubleMuon_2018D_21",
    # "DoubleMuon_2018D_22",
    # "DoubleMuon_2018D_23",
    # "DoubleMuon_2018D_24",
    # "DoubleMuon_2018D_25",
    # "DoubleMuon_2018D_26",
    # "DoubleMuon_2018D_27",
    # "DoubleMuon_2018D_28",
    # "DoubleMuon_2018D_29",
    # "DoubleMuon_2018D_30",
    # "DoubleMuon_2018D_31",
    # "DoubleMuon_2018D_32",
    # "DoubleMuon_2018D_33",
    # "DoubleMuon_2018D_34",
    # "DoubleMuon_2018D_35",
    # "DoubleMuon_2018D_36",
    # "DoubleMuon_2018D_37",
    # "DoubleMuon_2018D_38",
    # "DoubleMuon_2018D_39",
    # "DoubleMuon_2018D_40",
    # "DoubleMuon_2018D_41",
    # "DoubleMuon_2018D_42",
    # "DoubleMuon_2018D_43",
    # "DoubleMuon_2018D_44",
    # "DoubleMuon_2018D_45",
    # "DoubleMuon_2018D_46",
    # "DoubleMuon_2018D_47",
    # "DoubleMuon_2018D_48",
    # "DoubleMuon_2018D_49",
]

In [10]:
fileset = utilities.make_fileset(samples_data[0:60], 
                                 "llpNanoAOD_v2", 
                                 location_cfg="data_skimmed.yaml",
                                 max_files = -1,
                                )

In [11]:
out_data = {}
for i, data in enumerate(samples_data):
    
    print(f"Processing {data}")
    fileset_one_data = {samples_data[i]:fileset.get(samples_data[i])}
    
    output_data = runner.run(fileset_one_data, treename='Events', processor_instance=p)

    #Add this sample's output to the out variable
    out_data[data] = output_data["out"][data]

    #Save output to a file!!
    out_file_name = data + ".coffea"
    coffea.util.save(output_data,out_file_name)

Processing DoubleMuon_2018C
DoubleMuon_2018C is data. Not scaling histograms or cutflows.  6min  5.4s
Processing DoubleMuon_2018A_0
DoubleMuon_2018A_0 is data. Not scaling histograms or cutflows.4min 40.4s
Processing DoubleMuon_2018A_1
DoubleMuon_2018A_1 is data. Not scaling histograms or cutflows.4min 38.4s
Processing DoubleMuon_2018A_2
DoubleMuon_2018A_2 is data. Not scaling histograms or cutflows.3min 33.3s
